In [ ]:
# SPR 2026 – BERTimbau MAX_LEN=512 + Optuna Threshold Optimization
# Estratégia: Busca bayesiana (Optuna, 200 trials) dos threshold multipliers
# usando OOF predictions do model-v4 (all_results.pkl) para maximizar F1-macro.
# Base: resubmit_bertimbau_maxlen512_v0 (0.84027) – apenas calibração melhorada.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import f1_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
N_TRIALS    = 200
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

print(f'Device: {DEVICE}')
print(f'Weights dir: {weights_dir}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET & PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item


INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)


# ═══════════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR OOF PREDICTIONS DO MODEL-V4 → OTIMIZAR THRESHOLDS COM OPTUNA
# ═══════════════════════════════════════════════════════════════════════════════
artifacts_path = WEIGHTS_BASE / 'all_results.pkl'
print(f'Carregando OOF artifacts: {artifacts_path}')
with open(artifacts_path, 'rb') as f:
    all_results = pickle.load(f)

# Concatenar OOF predictions e labels de todos os folds
oof_probs  = []
oof_labels = []
for fold_key, fold_data in all_results.items():
    if isinstance(fold_data, dict):
        # Tentar diferentes nomes de chave
        probs  = fold_data.get('val_probs',  fold_data.get('oof_probs',  fold_data.get('probs', None)))
        labels = fold_data.get('val_labels', fold_data.get('oof_labels', fold_data.get('labels', None)))
        if probs is not None and labels is not None:
            oof_probs.append(np.array(probs))
            oof_labels.append(np.array(labels))

if oof_probs:
    oof_probs  = np.vstack(oof_probs)
    oof_labels = np.concatenate(oof_labels)
    print(f'OOF shape: {oof_probs.shape}, labels: {oof_labels.shape}')
    print(f'Baseline OOF F1-macro (argmax): {f1_score(oof_labels, oof_probs.argmax(axis=1), average="macro"):.5f}')
else:
    # Fallback: usar thresholds do solo_bert_artifacts
    print('AVISO: all_results.pkl não contém val_probs/oof_probs. Usando solo_bert_artifacts.pkl como fallback.')
    oof_probs  = None
    oof_labels = None


# Carregar temperatura do artifact original
solo_path = WEIGHTS_BASE / 'solo_bert_artifacts.pkl'
with open(solo_path, 'rb') as f:
    solo_artifacts = pickle.load(f)
BEST_TEMP = solo_artifacts['temperature']
print(f'Temperatura base: {BEST_TEMP:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPTUNA: OTIMIZAR THRESHOLD MULTIPLIERS (7 DIMENSÕES)
# ═══════════════════════════════════════════════════════════════════════════════
if oof_probs is not None:
    # Aplicar temperatura nos OOF probs antes de otimizar thresholds
    oof_calibrated = temperature_scale(oof_probs, BEST_TEMP)

    def objective(trial):
        thresholds = [
            trial.suggest_float(f't{c}', 0.05, 3.0) for c in range(NUM_CLASSES)
        ]
        preds = apply_thresholds(oof_calibrated, thresholds)
        return f1_score(oof_labels, preds, average='macro')

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_thresholds = [study.best_params[f't{c}'] for c in range(NUM_CLASSES)]
    print(f'\nMelhor F1-macro OOF (Optuna): {study.best_value:.5f}')
    print(f'Melhores thresholds: {[round(t, 4) for t in best_thresholds]}')
else:
    # Usar thresholds originais do artifact
    best_thresholds = list(solo_artifacts['thresholds'])
    print(f'Thresholds fallback: {[round(t, 4) for t in best_thresholds]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS DE TESTE
# ═══════════════════════════════════════════════════════════════════════════════
test_path = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df   = pd.read_csv(test_path)
test_texts = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Test samples: {len(test_df)}')

tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
dataset   = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 5-FOLD ENSEMBLE INFERENCE (model-v4)
# ═══════════════════════════════════════════════════════════════════════════════
config_path = weights_dir / 'model_config'
test_probs  = np.zeros((len(test_df), NUM_CLASSES))
folds_loaded = 0

for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        print(f'Fold {fold} não encontrado, pulando...')
        continue
    print(f'Carregando fold {fold}...', end=' ')
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    fold_probs  = predict(model, loader)
    test_probs += fold_probs
    folds_loaded += 1
    print(f'ok')
    del model, state_dict
    torch.cuda.empty_cache()

test_probs /= folds_loaded
print(f'{folds_loaded} folds carregados.')

# ═══════════════════════════════════════════════════════════════════════════════
# APLICAR TEMPERATURA + THRESHOLDS OTIMIZADOS → SUBMISSÃO
# ═══════════════════════════════════════════════════════════════════════════════
calibrated_probs = temperature_scale(test_probs, BEST_TEMP)
predictions      = apply_thresholds(calibrated_probs, best_thresholds)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')